# EGU26 Hydrological GNN Experiment Notebook (v3)

This notebook compares four structure-change candidates under a shared training and evaluation pipeline.

Compared models:
- Persistence-Residual GCN + FT-Transformer
- GCN + FT-Transformer
- Delta-Chl GCN + GRU
- Delta-Chl GCN + Transformer

Selection rule:
- prioritize `val` metrics for model selection
- use `test` metrics for final confirmation
- inspect both overall metrics and high-chl metrics


## 1. Environment Setup
Use this notebook in Google Colab or in the local repository checkout.


In [ ]:
import os
import sys
from pathlib import Path

if os.path.exists('/content'):
    REPO_ROOT = Path('/content/EGU26-SWAT-GNN')
    if not REPO_ROOT.exists():
        !git clone https://github.com/kona0107/EGU26-SWAT-GNN.git /content/EGU26-SWAT-GNN
    MODULE_PATH = REPO_ROOT / 'script' / 'src' / 'gnn_project'
    DATA_PATH = REPO_ROOT / 'script' / 'data'
else:
    cwd = Path.cwd()
    if (cwd / 'script' / 'src' / 'gnn_project').exists():
        REPO_ROOT = cwd
    else:
        REPO_ROOT = cwd.parents[2]
    MODULE_PATH = REPO_ROOT / 'script' / 'src' / 'gnn_project'
    DATA_PATH = REPO_ROOT / 'script' / 'data'

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

OUTLET_CSV = DATA_PATH / 'outlet.csv'
UPSTREAM_CSV = DATA_PATH / 'upstream.csv'
RESULTS_DIR = REPO_ROOT / 'results' / 'train_v3'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT   :', REPO_ROOT)
print('MODULE_PATH :', MODULE_PATH)
print('DATA_PATH   :', DATA_PATH)
print('RESULTS_DIR :', RESULTS_DIR)
print('Outlet CSV  :', OUTLET_CSV.exists())
print('Upstream CSV:', UPSTREAM_CSV.exists())


In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')

if os.path.exists('/content'):
    !pip install torch_geometric -q
    !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html -q


## 2. Imports and Shared Config


In [ ]:
import copy
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader

from data.dataset import (
    CHL_IDX,
    FEATURE_DIM,
    OUTLET_IDX,
    RAW_DIM,
    load_real_data,
    prepare_and_split_data,
)
from models.advanced_hybrid import (
    GCNFTTransformerModel,
    PersistenceResidualGCNFTTransformerModel,
)
from models.st_gcn import SpatioTemporalHybridGNN

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

SEED = 42
LOOKBACK = 14
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
BATCH_SIZE = 16
TEMPORAL_HIDDEN = 64
GCN_HIDDEN = 32
NUM_EPOCHS = 200
LR = 5e-4
PATIENCE = 30
HIGH_THRESHOLD_RAW = 80.0
HIGH_WEIGHT = 2.0
FT_NUM_LAYERS = 2
FT_NHEAD = 4
FT_DROPOUT = 0.1


## 3. Data Loading and Split


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

raw_features, dates = load_real_data(str(OUTLET_CSV), str(UPSTREAM_CSV))
print(f'Total dates      : {len(dates)}')
print(f'raw_features     : {raw_features.shape}  -> expected [T, 29, {RAW_DIM}]')
print(f'date range       : {dates[0]} ~ {dates[-1]}')

train_ds, val_ds, test_ds, scaler_out, scaler_target = prepare_and_split_data(
    raw_features,
    outlet_node_idx=OUTLET_IDX,
    lookback_window=LOOKBACK,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    apply_log1p=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

total_steps = len(dates)
train_end = int(total_steps * TRAIN_RATIO)
val_end = train_end + int(total_steps * VAL_RATIO)
train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]

bx, by, bidx = next(iter(train_loader))
print(f'Train / Val / Test samples: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')
print(f'Batch X shape             : {bx.shape}')
print(f'Batch y shape             : {by.shape}')


## 4. Directed Graph Definition


In [ ]:
edges_list = [
    (20, 5), (5, 23), (21, 24), (22, 24), (24, 23), (23, 25), (4, 25), (19, 25),
    (25, 27), (15, 3), (3, 27), (6, 27), (10, 27), (16, 27), (27, 0), (0, 28),
    (26, 2), (2, 1), (12, 1), (1, 28), (7, 28), (8, 28), (9, 28),
]
source = torch.tensor([edge[0] for edge in edges_list], dtype=torch.long)
target = torch.tensor([edge[1] for edge in edges_list], dtype=torch.long)
edge_index = torch.stack([source, target], dim=0).to(DEVICE)

print('edge_index shape:', edge_index.shape)


## 5. Training and Evaluation Utilities


In [ ]:
def make_weighted_mse_loss(high_threshold_raw=80.0, high_weight=2.0):
    threshold_scaled = scaler_target.transform(
        np.array([[np.log1p(high_threshold_raw)]], dtype=np.float32)
    )[0, 0]

    def weighted_mse(pred, target):
        weights = torch.ones_like(target)
        weights = torch.where(target >= threshold_scaled, high_weight, weights)
        return torch.mean(weights * (pred - target) ** 2)

    return weighted_mse


def inverse_target(arr_2d):
    return np.expm1(scaler_target.inverse_transform(arr_2d)).ravel()


def compute_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
    }


def compute_high_metrics(y_true, y_pred, threshold=80.0):
    mask = y_true >= threshold
    if mask.sum() == 0:
        return {
            'high_count': 0,
            'high_R2': np.nan,
            'high_RMSE': np.nan,
            'high_MAE': np.nan,
        }
    metrics = compute_metrics(y_true[mask], y_pred[mask])
    return {
        'high_count': int(mask.sum()),
        'high_R2': metrics['R2'],
        'high_RMSE': metrics['RMSE'],
        'high_MAE': metrics['MAE'],
    }


def build_model(config):
    family = config['model_family']

    if family == 'gcn_ft_transformer':
        return GCNFTTransformerModel(
            in_features=FEATURE_DIM,
            lookback_window=LOOKBACK,
            temporal_hidden=TEMPORAL_HIDDEN,
            gcn_hidden=GCN_HIDDEN,
            num_layers=FT_NUM_LAYERS,
            nhead=FT_NHEAD,
            dropout=FT_DROPOUT,
        ).to(DEVICE)

    if family == 'persistence_residual_gcn_ft_transformer':
        return PersistenceResidualGCNFTTransformerModel(
            in_features=FEATURE_DIM,
            lookback_window=LOOKBACK,
            temporal_hidden=TEMPORAL_HIDDEN,
            gcn_hidden=GCN_HIDDEN,
            num_layers=FT_NUM_LAYERS,
            nhead=FT_NHEAD,
            dropout=FT_DROPOUT,
            chl_feature_idx=CHL_IDX,
        ).to(DEVICE)

    if family == 'delta_gcn_gru':
        return SpatioTemporalHybridGNN(
            in_features=FEATURE_DIM,
            temporal_hidden=TEMPORAL_HIDDEN,
            gcn_hidden=GCN_HIDDEN,
            temporal_type='gru',
        ).to(DEVICE)

    if family == 'delta_gcn_transformer':
        return SpatioTemporalHybridGNN(
            in_features=FEATURE_DIM,
            temporal_hidden=TEMPORAL_HIDDEN,
            gcn_hidden=GCN_HIDDEN,
            temporal_type='transformer',
        ).to(DEVICE)

    raise ValueError(f"Unknown model family: {family}")


def prepare_loss_targets(config, x_batch, y_batch, preds):
    target_mode = config.get('target_mode', 'absolute')
    last_outlet_chl = x_batch[:, -1, OUTLET_IDX, CHL_IDX].unsqueeze(-1)

    if target_mode == 'delta':
        target = y_batch.view_as(preds) - last_outlet_chl
        pred_absolute = preds + last_outlet_chl
    else:
        target = y_batch.view_as(preds)
        pred_absolute = preds

    return target, pred_absolute


@torch.no_grad()
def collect_predictions(model, loader, split_dates, config):
    model.eval()
    y_true_scaled_all = []
    y_pred_scaled_all = []
    target_indices = []

    for x_batch, y_batch, target_idx in loader:
        x_batch = x_batch.to(DEVICE)
        preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
        _, pred_absolute = prepare_loss_targets(config, x_batch, y_batch.to(DEVICE), preds)

        y_true_scaled_all.append(y_batch.numpy())
        y_pred_scaled_all.append(pred_absolute.cpu().numpy())
        target_indices.extend(target_idx.numpy().tolist())

    y_true_scaled = np.concatenate(y_true_scaled_all, axis=0)
    y_pred_scaled = np.concatenate(y_pred_scaled_all, axis=0)
    y_true_raw = inverse_target(y_true_scaled)
    y_pred_raw = inverse_target(y_pred_scaled)

    pred_df = pd.DataFrame({
        'target_idx': target_indices,
        'date': [pd.Timestamp(split_dates[idx]) for idx in target_indices],
        'y_true': y_true_raw,
        'y_pred': y_pred_raw,
    })
    pred_df['abs_err'] = np.abs(pred_df['y_pred'] - pred_df['y_true'])
    return y_true_raw, y_pred_raw, pred_df


def train_and_evaluate(config):
    set_seed(SEED)
    model = build_model(config)
    criterion = make_weighted_mse_loss(
        high_threshold_raw=config['high_threshold'],
        high_weight=config['high_weight'],
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

    best_val_loss = float('inf')
    best_state = None
    patience_count = 0
    history = []

    start_time = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0

        for x_batch, y_batch, _ in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
            target_for_loss, _ = prepare_loss_targets(config, x_batch, y_batch, preds)
            loss = criterion(preds, target_for_loss)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * len(x_batch)

        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for x_batch, y_batch, _ in val_loader:
                x_batch = x_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
                target_for_loss, _ = prepare_loss_targets(config, x_batch, y_batch, preds)
                loss = criterion(preds, target_for_loss)
                val_loss_sum += loss.item() * len(x_batch)

        val_loss = val_loss_sum / len(val_loader.dataset)
        scheduler.step(val_loss)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'lr': optimizer.param_groups[0]['lr'],
        })

        if epoch == 1 or epoch % 20 == 0:
            print(f"[{config['name']}] epoch={epoch:3d} train={train_loss:.4f} val={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"[{config['name']}] Early stopping at epoch {epoch}, best val={best_val_loss:.4f}")
                break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)

    tr_true, tr_pred, train_pred_df = collect_predictions(model, train_loader, train_dates, config)
    va_true, va_pred, val_pred_df = collect_predictions(model, val_loader, val_dates, config)
    te_true, te_pred, test_pred_df = collect_predictions(model, test_loader, test_dates, config)

    train_metrics = compute_metrics(tr_true, tr_pred)
    val_metrics = compute_metrics(va_true, va_pred)
    test_metrics = compute_metrics(te_true, te_pred)
    val_high = compute_high_metrics(va_true, va_pred, threshold=config['high_threshold'])
    test_high = compute_high_metrics(te_true, te_pred, threshold=config['high_threshold'])

    elapsed = time.time() - start_time
    result = {
        'name': config['name'],
        'model_family': config['model_family'],
        'target_mode': config['target_mode'],
        'high_threshold': config['high_threshold'],
        'high_weight': config['high_weight'],
        'train_R2': train_metrics['R2'],
        'train_RMSE': train_metrics['RMSE'],
        'train_MAE': train_metrics['MAE'],
        'val_R2': val_metrics['R2'],
        'val_RMSE': val_metrics['RMSE'],
        'val_MAE': val_metrics['MAE'],
        'val_high_count': val_high['high_count'],
        'val_high_R2': val_high['high_R2'],
        'val_high_RMSE': val_high['high_RMSE'],
        'val_high_MAE': val_high['high_MAE'],
        'test_R2': test_metrics['R2'],
        'test_RMSE': test_metrics['RMSE'],
        'test_MAE': test_metrics['MAE'],
        'test_high_count': test_high['high_count'],
        'test_high_R2': test_high['high_R2'],
        'test_high_RMSE': test_high['high_RMSE'],
        'test_high_MAE': test_high['high_MAE'],
        'best_val_loss': best_val_loss,
        'elapsed_sec': elapsed,
    }

    artifacts = {
        'history': history_df,
        'train_pred': train_pred_df,
        'val_pred': val_pred_df,
        'test_pred': test_pred_df,
    }
    return result, artifacts, model


## 6. Experiment Definitions


In [ ]:
experiment_configs = [
    {
        'name': 'persistence_residual_gcn_ft_transformer',
        'model_family': 'persistence_residual_gcn_ft_transformer',
        'target_mode': 'absolute',
        'high_threshold': HIGH_THRESHOLD_RAW,
        'high_weight': HIGH_WEIGHT,
    },
    {
        'name': 'gcn_ft_transformer',
        'model_family': 'gcn_ft_transformer',
        'target_mode': 'absolute',
        'high_threshold': HIGH_THRESHOLD_RAW,
        'high_weight': HIGH_WEIGHT,
    },
    {
        'name': 'delta_chl_gcn_gru',
        'model_family': 'delta_gcn_gru',
        'target_mode': 'delta',
        'high_threshold': HIGH_THRESHOLD_RAW,
        'high_weight': HIGH_WEIGHT,
    },
    {
        'name': 'delta_chl_gcn_transformer',
        'model_family': 'delta_gcn_transformer',
        'target_mode': 'delta',
        'high_threshold': HIGH_THRESHOLD_RAW,
        'high_weight': HIGH_WEIGHT,
    },
]

pd.DataFrame(experiment_configs)


## 7. Run Experiments


In [ ]:
all_results = []
all_artifacts = {}

for idx, config in enumerate(experiment_configs, start=1):
    print(f"\n===== [{idx}/{len(experiment_configs)}] RUN: {config['name']} =====")
    result, artifacts, model = train_and_evaluate(config)
    all_results.append(result)
    all_artifacts[config['name']] = artifacts

    artifacts['history'].to_csv(RESULTS_DIR / f"history_{config['name']}.csv", index=False)
    artifacts['train_pred'].to_csv(RESULTS_DIR / f"train_pred_{config['name']}.csv", index=False)
    artifacts['val_pred'].to_csv(RESULTS_DIR / f"val_pred_{config['name']}.csv", index=False)
    artifacts['test_pred'].to_csv(RESULTS_DIR / f"test_pred_{config['name']}.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(
    ['val_R2', 'val_high_MAE', 'test_R2'],
    ascending=[False, True, False],
)
results_df.to_csv(RESULTS_DIR / 'summary_train_v3.csv', index=False)
display(results_df)
print('Saved summary to:', RESULTS_DIR / 'summary_train_v3.csv')


## 8. Comparison Views


In [ ]:
summary_cols = [
    'name',
    'train_R2', 'val_R2', 'test_R2',
    'val_high_MAE', 'test_high_MAE',
    'val_high_RMSE', 'test_high_RMSE',
    'best_val_loss', 'elapsed_sec',
]

print('=== Ranked by val_R2, then val_high_MAE ===')
display(results_df[summary_cols])

print('=== Ranked by test_high_MAE ===')
display(results_df.sort_values(['test_high_MAE', 'test_R2'], ascending=[True, False])[summary_cols])


## 9. Visualization


In [ ]:
ordered_names = results_df['name'].tolist()

plt.figure(figsize=(16, 6))
best_name = ordered_names[0]
best_pred_df = all_artifacts[best_name]['test_pred']
plt.plot(best_pred_df['y_true'].values, label='Observed', linewidth=2.2, color='black')

for name in ordered_names:
    pred_df = all_artifacts[name]['test_pred']
    plt.plot(pred_df['y_pred'].values, label=name, linestyle='--')

plt.title('Test Prediction Curve Comparison (Raw Scale)')
plt.xlabel('Test Time Step')
plt.ylabel('Chl-a Concentration (ug/L)')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 8))
for name in ordered_names:
    pred_df = all_artifacts[name]['test_pred']
    plt.scatter(pred_df['y_true'], pred_df['y_pred'], alpha=0.55, label=name)

max_value = max(all_artifacts[name]['test_pred']['y_true'].max() for name in ordered_names)
plt.plot([0, max_value], [0, max_value], 'k--', linewidth=1.2, label='Ideal')
plt.title('Observed vs Predicted on Test Set')
plt.xlabel('Observed Chl-a (ug/L)')
plt.ylabel('Predicted Chl-a (ug/L)')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 5))
for name in ordered_names:
    peak_df = all_artifacts[name]['test_pred']
    peak_df = peak_df[peak_df['y_true'] >= HIGH_THRESHOLD_RAW].reset_index(drop=True)
    if len(peak_df) == 0:
        continue
    if name == ordered_names[0]:
        plt.plot(peak_df['y_true'].values, label='Observed Peaks', linewidth=2.2, color='black')
    plt.plot(peak_df['y_pred'].values, label=name, linestyle='--')

plt.title(f'Peak Region Comparison (y_true >= {HIGH_THRESHOLD_RAW:.0f})')
plt.xlabel('Peak Sample Index')
plt.ylabel('Chl-a Concentration (ug/L)')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 10. Save Ranked Recommendations
Use the `val`-ranked summary as the main selection view, and confirm the final candidate with the `test` metrics.
